# Consultas usando OQL/OQO

OQL (OpenAlex Query Language), OQO (OpenAlex Query Objects)

https://help.openalex.org/api/oql/

También se usa un nuevo formato de queries de consulta por eje temático usando `queries_honduras_ET.json`.

In [68]:
import os
import time
import requests
import json
import re
import pandas as pd

from dotenv import load_dotenv

In [69]:
load_dotenv()
OA_API_KEY = os.getenv("OPENALEX_API_KEY")

In [70]:
#file_str = "../data/queries_honduras_ET.json"
file_str = "../data/queries_ecuador.json"
with open(file_str, "r", encoding="utf-8") as file:
    queries_honduras = json.load(file)

print(queries_honduras['meta'])

{'country': 'Ecuador', 'start_year': 2019, 'end_year': 2026, 'description': 'This dataset contains a set of queries related to the IDB Group Country Strategy with Ecuador 2026-2030. Each query is designed to retrieve studies for each thematic axis (eje temático) defined by experts', 'country_terms': ['Ecuador', 'Ecuadorian', 'Ecuadorians', 'ecuatoriano', 'ecuatoriana', 'ecuatorianos', 'ecuatorianas', 'afroecuatoriano', 'afroecuatoriana', 'afroecuatorianos', 'afroecuatorianas']}


Construir consulta con OQL

In [71]:
from urllib.parse import quote

OA_BASE = "https://api.openalex.org/"
OA_HEADERS = {"Authorization": f"Bearer {OA_API_KEY}"}

allowed_types = ["article", "report", "conference-paper", "book", "book-chapter",
                "dissertation", "conference-abstract"]
country_terms = queries_honduras['meta']['country_terms']
start_year = queries_honduras['meta']['start_year']
end_year = queries_honduras['meta']['end_year']

In [72]:
def build_oql(keywords, start_year, end_year, country_terms, allowed_types, 
                field="title-abstract-keywords", stem=True):
    ''' Build OQL: works between years, with allowed types, 
    containing any country term AND any keyword (stemmed) '''
    clean = lambda text: text.replace('"', '').strip()
    if stem:
        kw_search = " OR ".join(f'stemmed "{clean(kw)}"' for kw in keywords)
    else:
        kw_search = " OR ".join(f'"{clean(kw)}"' for kw in keywords)
    country_search = " OR ".join(clean(term) if " " not in clean(term) else f'stemmed "{clean(term)}"'
                                for term in country_terms)
    types = " or ".join(allowed_types)
    oql = (f"get works where year >= ({start_year}) and year <= ({end_year})"
           f" and type is ({types})"
           f" and {field} has (({country_search}) AND ({kw_search}))")
    return oql

def build_openalex_payload(oql, per_page=100, cursor="*", select=None):
    ''' Body for POST request to OpenAlex API root '''
    payload = {"oql": oql, "per_page": per_page, "cursor": cursor}
    if select:
        payload["select"] = ",".join(select)
    return payload

## Validación de las consultas

Se usa el endpoint gratuito `https://api.openalex.org/query/oql/<oql>`, que traduce y valida la OQL sin ejecutarla. El warning `url_not_representable` es esperado: la consulta mezcla búsquedas con y sin stemming en un OR, y eso no tiene equivalente en la URL clásica.

In [73]:
def validate_oql(oql):
    ''' Validate OQL with the free translate endpoint /query/oql (does not run the query) '''
    req = requests.get(OA_BASE + "query/oql/" + quote(oql, safe=""), headers=OA_HEADERS)
    try:
        body = req.json()
    except ValueError:
        print(f"Error: {req.status_code}")
        return False, None
    validation = body.get("validation", {})
    valid = req.status_code == 200 and validation.get("valid", False)
    if not valid:
        print(f"Invalid OQL ({req.status_code}):", validation.get("errors", body))
    # url_not_representable is expected: OQL has no classic URL equivalent
    warnings = [w for w in validation.get("warnings", []) if w.get("type") != "url_not_representable"]
    if warnings:
        print("Warnings:", warnings)
    return valid, body

In [74]:
validation_rows = []
for thematic in queries_honduras['thematics']:
    oql = build_oql(thematic['keywords'], start_year, end_year, country_terms, 
                    allowed_types, field="title-abstract",stem=False)
    valid, body = validate_oql(oql)
    validation = body.get('validation', {}) if body else {}
    validation_rows.append({
        "thematic_id": thematic['thematic_id'],
        "thematic_axis": thematic['thematic_axis'],
        "n_keywords": len(thematic['keywords']),
        "len_oql": len(oql),
        "valid": valid,
        "errors": validation.get('errors'),
        "warnings": [w.get('type') for w in validation.get('warnings', [])],
        "oql_rendered": body.get('oql') if body else None
    })
    time.sleep(0.5)

validation_df = pd.DataFrame(validation_rows)

In [75]:
validation_df

,thematic_id,thematic_axis,n_keywords,len_oql,valid,errors,warnings,oql_rendered
0,1111,Entorno de negocios e inversión,29,1104,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...
1,1112,"Productividad, innovación y cadenas productivas",18,847,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...
2,1113,Regulación laboral,25,1021,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...
3,1121,Logística e infraestructura de transporte,34,1244,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...
4,1122,Comercio exterior e integración comercial,22,933,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...
5,1131,Conectividad e infraestructura digital,28,1186,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...
6,1141,"Mercados financieros, financiamiento productiv...",36,1268,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...
7,1142,Financiamiento de vivienda,14,744,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...
8,1211,Sistema eléctrico y distribución de electricidad,29,1087,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...
9,1221,Resiliencia del sistema eléctrico,37,1701,True,[],[],get works from 2019–2026 with ((Ecuador OR Ecu...


In [76]:
validation_df['oql_rendered'].values

<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     'get works from 2019–2026 with ((Ecuador OR Ecuadorian OR Ecuadorians OR ecuatoriano OR ecuatoriana OR ecuatorianos OR ecuatorianas OR afroecuatoriano OR afroecuatoriana OR afroecuatorianos OR afroecuatorianas) AND ("investment climate" OR "clima de inversión" OR "business environment" OR "entorno empresarial" OR "entorno de negocios" OR "private investment" OR "inversión privada" OR "

In [77]:
assert validation_df['valid'].all(), "Hay consultas OQL inválidas, revisar antes de ejecutar"

# OQL renderizada por OpenAlex para la primera temática
print(validation_df.loc[0, 'oql_rendered'])

get works from 2019–2026 with ((Ecuador OR Ecuadorian OR Ecuadorians OR ecuatoriano OR ecuatoriana OR ecuatorianos OR ecuatorianas OR afroecuatoriano OR afroecuatoriana OR afroecuatorianos OR afroecuatorianas) AND ("investment climate" OR "clima de inversión" OR "business environment" OR "entorno empresarial" OR "entorno de negocios" OR "private investment" OR "inversión privada" OR "investment barriers" OR "barreras a la inversión" OR "regulatory barriers" OR "barreras regulatorias" OR "business regulation" OR "regulación empresarial" OR "regulatory reform" OR "reforma regulatoria" OR "regulatory quality" OR "calidad regulatoria" OR "investment promotion" OR "promoción de inversiones" OR "foreign direct investment" OR "inversión extranjera directa" OR "competitiveness" OR "competitividad" OR "marco legal" OR "legal framework" OR "incentivos a las inversiones" OR "investment incentives" OR "seguridad jurídica" OR "legal certainty")) in the title or abstract where type is ([article](art

## Consulta a OpenAlex

`POST` a la raíz de la API con la OQL en el body. La paginación usa `cursor` siguiendo `meta.next_cursor`. La salida mantiene el formato `{meta, results}` de la URL clásica.

In [78]:
def openalex_request(oql, per_page=100, max_results=10_000, select=None):
    ''' Request to OpenAlex API with OQL, paginating with cursor '''
    data = None
    results = []
    cursor = "*"
    pages, total_cost, total_time_ms = 0, 0, 0
    complete = True

    while cursor is not None and len(results) < max_results:
        payload = build_openalex_payload(oql, per_page, cursor, select)
        req = requests.post(OA_BASE, json=payload, headers=OA_HEADERS)
        if req.status_code != 200:
            try:
                print(f"Error: {req.status_code}", req.json().get("validation"))
            except ValueError:
                print(f"Error: {req.status_code}")
            if data is None:
                return None # First page failed
            complete = False # Keep pages already fetched
            break

        data_page = req.json()
        if data is None:
            data = data_page
            if data['meta']['count'] > max_results:
                print(f"Warning: Query has {data['meta']['count']} results, only {max_results} will be fetched")

        results.extend(data_page['results'])
        total_cost += data_page['meta']['cost_usd']
        total_time_ms += data_page['meta']['db_response_time_ms']
        cursor = data_page['meta']['next_cursor']
        pages += 1
        if cursor is not None:
            time.sleep(0.5)

    # Cursor paging sorted by relevance can repeat a few works between pages
    n_fetched = len(results)
    if all('id' in result for result in results):
        results = list({result['id']: result for result in results}.values())

    data['results'] = results
    data['meta']['pages'] = pages
    data['meta']['cost_usd'] = total_cost
    data['meta']['db_response_time_ms'] = total_time_ms
    data['meta']['next_cursor'] = cursor
    data['meta']['complete'] = complete
    data['meta']['duplicates_removed'] = n_fetched - len(results)
    return data

Probar request

In [79]:
thematic = queries_honduras['thematics'][2]
oql = build_oql(thematic['keywords'], start_year, end_year, country_terms, allowed_types)
openalex_data = openalex_request(oql)

print(json.dumps({k: openalex_data['meta'][k] for k in ['count', 'pages', 'cost_usd', 'complete', 'duplicates_removed']}, indent=4))
print(openalex_data['meta']['x_query']['oql'])
print(len(openalex_data['results']))

{
    "count": 355,
    "pages": 4,
    "cost_usd": 0.004,
    "complete": true,
    "duplicates_removed": 0
}
get works from 2019–2026 about ((Ecuador OR Ecuadorian OR Ecuadorians OR ecuatoriano OR ecuatoriana OR ecuatorianos OR ecuatorianas OR afroecuatoriano OR afroecuatoriana OR afroecuatorianos OR afroecuatorianas) AND (stemmed "labor regulation" OR stemmed "labour regulation" OR stemmed "regulación laboral" OR stemmed "labor market regulation" OR stemmed "labour market regulation" OR stemmed "regulación del mercado laboral" OR stemmed "employment regulation" OR stemmed "regulación del empleo" OR stemmed "labor costs" OR stemmed "labour costs" OR stemmed "costos laborales" OR stemmed "employment protection" OR stemmed "protección del empleo" OR stemmed "labor market flexibility" OR stemmed "labour market flexibility" OR stemmed "flexibilidad del mercado laboral" OR stemmed "capital humano capacitado" OR stemmed "skilled human capital" OR stemmed "skilled workers" OR stemmed "traba

### Helpers

In [80]:
interest_vars = ["id", "doi", "display_name", "relevance_score",
                 "publication_year", "publication_date", "language", 
                 "primary_location.landing_page_url",
                 "primary_location.source.id", "primary_location.source.host_organization_name",
                 "primary_location.source.display_name", "primary_location.raw_source_name", 
                 "primary_location.source.type", "primary_location.raw_type", "type",
                 "primary_location.is_oa", "open_access.is_oa", "open_access.oa_status", "open_access.oa_url",
                 "fwci", "cited_by_count", "citation_normalized_percentile.value",
                 "primary_topic.id", "primary_topic.display_name", 
                 "primary_topic.subfield.id", "primary_topic.subfield.display_name",
                 "primary_topic.field.id", "primary_topic.field.display_name",
                 "primary_topic.domain.id", "primary_topic.domain.display_name",
                 "updated_date"
]

# Inside "authorships" OBJECT list
authors_vars = ["author.id", "author.display_name"]
# Inside "authorships[n].institutions" list
institutions_vars = ["id", "display_name", "country_code", "type"]

abstract_object = "abstract_inverted_index" # Abstract as set of words with list of their positions

keywords_object = "keywords" # List of keyword objects, each with "id", "display_name" and "score"

In [81]:
# Build abstract/description
def build_abstract(abstract_inverted_index):
    if isinstance(abstract_inverted_index, dict):
        index_items= abstract_inverted_index.items()
        index_expand = {i: k for k, v in index_items for i in v}
        index_expand = sorted(index_expand.items(), key=lambda x: x[0])
        abstract = ' '.join([word for position, word in index_expand])

        # Filter printable chars
        abstract =  ''.join(ch for ch in abstract if ch.isprintable() or ch.isspace())
        return abstract
    else:
        return None

In [82]:
clean_value = lambda value: str(value or '').replace(';', ',') # ";" inside a value would break the alignment

def build_authors(authorships):
    ''' Authors fields joined by ";" keeping the authorship order '''
    authorships = authorships if isinstance(authorships, list) else []
    authors = [authorship.get('author') or {} for authorship in authorships]
    return {
        f'authorships.{au_var}': ';'.join(clean_value(author.get(au_var.split('.', 1)[1])) for author in authors)
        for au_var in authors_vars
    }

def build_institutions(authorships):
    ''' Unique institutions (by id, keeping order) with aligned fields joined by ";" '''
    authorships = authorships if isinstance(authorships, list) else []
    institutions = {}
    for authorship in authorships:
        for inst in authorship.get('institutions') or []:
            institutions.setdefault(inst.get('id'), inst)
    return {
        f'authorships.institutions.{inst_var}': ';'.join(clean_value(inst.get(inst_var)) for inst in institutions.values())
        for inst_var in institutions_vars
    }

def build_keywords(keywords):
    ''' Keywords display names joined by ";" '''
    keywords = keywords if isinstance(keywords, list) else []
    return ';'.join(keyword.get('display_name') or '' for keyword in keywords)

def build_studies_df(results, thematic_id):
    ''' Studies dataframe with interest variables, authors, institutions, abstract and keywords '''
    extra_vars = ([f'authorships.{au_var}' for au_var in authors_vars]
                  + [f'authorships.institutions.{inst_var}' for inst_var in institutions_vars]
                  + ['abstract', 'keywords'])

    # Nested objects are built apart (json_normalize would expand the abstract index into one column per word)
    nested_vars = ['authorships', abstract_object, keywords_object]
    studies = pd.json_normalize(
        [{k: v for k, v in result.items() if k not in nested_vars} for result in results]
        )
    studies = studies.reindex(columns=interest_vars) # Missing columns as NaN

    extra = pd.DataFrame([
        {**build_authors(result.get('authorships')),
         **build_institutions(result.get('authorships')),
        'abstract': build_abstract(result.get(abstract_object)),
        'keywords': build_keywords(result.get(keywords_object))}
        for result in results
    ], columns=extra_vars)

    studies = pd.concat([studies, extra], axis=1)
    studies['thematic_id'] = thematic_id
    return studies

In [83]:
# Thematics dataframe
thematics_df = pd.DataFrame(queries_honduras['thematics'])
thematics_df['n_keywords'] = thematics_df['keywords'].apply(len)
thematics_df['oql'] = thematics_df['keywords'].apply(
    lambda keywords: build_oql(keywords, start_year, end_year, country_terms, allowed_types)
    )
thematics_df['keywords'] = thematics_df['keywords'].apply('; '.join)

## Todas las temáticas

Consulta OpenAlex para cada temática de `queries_honduras_ET.json`. El resultado de cada `openalex_request()` se guarda en `honduras_backup/thematic_{thematic_id}.json`. `studies_df` contiene los estudios con `thematic_id`, y las variables de temática quedan en `thematics_df`.

In [98]:
backup_dir = "honduras_backup"
os.makedirs(backup_dir, exist_ok=True)

total_cost, total_db_time_ms = 0, 0
studies_list = []
start_time = time.perf_counter()

print('Total thematics:', len(queries_honduras['thematics']))
for thematic in queries_honduras['thematics'][2:]:
    thematic_id = thematic['thematic_id']
    print(f"Thematic {thematic_id}: {thematic['thematic_axis']}")
    oql = build_oql(thematic['keywords'], start_year, end_year, 
                    ["Ecuador"], allowed_types,
                    field="title-abstract", stem=False)
    q_data = openalex_request(oql)
    if q_data is None:
        print(f"Warning: thematic {thematic_id} failed, skipping")
        continue

    total_cost += q_data['meta']['cost_usd']
    total_db_time_ms += q_data['meta']['db_response_time_ms']

    with open(os.path.join(backup_dir, f"thematic_{thematic_id}.json"), "w", encoding="utf-8") as f: # Save backup
        json.dump(q_data, f, indent=4, ensure_ascii=False)

    studies_list.append(build_studies_df(q_data['results'], thematic_id))
    time.sleep(0.5)

total_time_s = time.perf_counter() - start_time

studies_df = pd.concat(studies_list, ignore_index=True)

print('Total OpenAlex cost USD:', round(total_cost, 4))
print('Total time (s):', round(total_time_s, 1))
print('Total OpenAlex DB time (s):', round(total_db_time_ms / 1000, 1))
print('Total studies:', len(studies_df))

Total thematics: 22
Thematic 1113: Regulación laboral
Thematic 1121: Logística e infraestructura de transporte
Thematic 1122: Comercio exterior e integración comercial
Thematic 1131: Conectividad e infraestructura digital
Thematic 1141: Mercados financieros, financiamiento productivo e inclusión financiera
Thematic 1142: Financiamiento de vivienda
Thematic 1211: Sistema eléctrico y distribución de electricidad
Thematic 1221: Resiliencia del sistema eléctrico
Thematic 2111: Empleabilidad e inserción laboral
Thematic 2121: Atención primaria de salud
Thematic 2131: Agua potable y saneamiento
Thematic 3111: Gestión fiscal y macroeconomía
Thematic 3112: Administración tributaria
Thematic 3113: Gestión de inversión y riesgos fiscales
Thematic 3121: Compras y contratación pública
Thematic 3122: Eficiencia y transparencia del gasto público
Thematic 3211: Control y gestión migratoria
Thematic 3221: Lavado de activos y flujos financieros ilícitos
Thematic 3231: Prevención del reclutamiento y vio

In [93]:
# Read backup files and merge into a single dataframe
studies_list = []
ids = [thematic['thematic_id'] for thematic in queries_honduras['thematics']]
for id in ids:
    with open(f"honduras_backup/thematic_{id}.json", "r", encoding="utf-8") as f:
        q_data = json.load(f)
    studies_list.append(build_studies_df(q_data['results'], id))
studies_df = pd.concat(studies_list, ignore_index=True)

In [94]:
studies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22319 entries, 0 to 22318
Data columns (total 40 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              22319 non-null  str    
 1   doi                                             16265 non-null  str    
 2   display_name                                    22318 non-null  str    
 3   relevance_score                                 22319 non-null  float64
 4   publication_year                                22319 non-null  int64  
 5   publication_date                                22319 non-null  str    
 6   language                                        22316 non-null  str    
 7   primary_location.landing_page_url               22319 non-null  str    
 8   primary_location.source.id                      20543 non-null  str    
 9   primary_location.source.host_organization_name  10

In [18]:
studies_df.sample(5, random_state=42)

,id,doi,display_name,relevance_score,publication_year,publication_date,language,primary_location.landing_page_url,primary_location.source.id,primary_location.source.host_organization_name,...,updated_date,authorships.author.id,authorships.author.display_name,authorships.institutions.id,authorships.institutions.display_name,authorships.institutions.country_code,authorships.institutions.type,abstract,keywords,thematic_id
1374,https://openalex.org/W3167457633,https://doi.org/10.1007/s11205-021-02730-7,Does Financial Inclusion Diminish Poverty and ...,57.875793,2021,2021-06-09,en,https://doi.org/10.1007/s11205-021-02730-7,https://openalex.org/S112952035,Springer Science+Business Media,...,2026-10-30T12:39:25.321706,https://openalex.org/A5019430484;https://opena...,Eduardo Polloni‐Silva;Naijela Janaina Costa Si...,https://openalex.org/I177909021;https://openal...,Universidade Federal de São Carlos;Universidad...,BR;BR,education;education,NaN,financial inclusion;poverty;income inequality;...,321
1080,https://openalex.org/W7196948433,https://doi.org/10.18687/laccei2026.1.1.1513,Structural Determinants of Tax Revenue: A Bala...,31.097828,2026,2026-01-01,en,https://doi.org/10.18687/laccei2026.1.1.1513,https://openalex.org/S4363608766,NaN,...,2026-10-29T09:56:32.521200,https://openalex.org/A5145707442,German Salgado Valle,,,,,This study examines the structural determinant...,tax revenue;structural factors;tax revenue for...,210
1519,https://openalex.org/W3176948599,https://doi.org/10.1080/21632324.2021.1934022,A love-hate relationship: An ethnographic stud...,139.982800,2021,2021-06-18,en,https://doi.org/10.1080/21632324.2021.1934022,https://openalex.org/S4210192817,Taylor & Francis,...,2026-09-24T07:57:37.273135,https://openalex.org/A5016684402;https://opena...,Pablo Lamiño;Amy E. Boren-Alpízar;Sarahí Moral...,https://openalex.org/I12315562;https://openale...,Texas Tech University;Zamorano Pan-American Ag...,US;HN,education;education,Honduras is a Latin-American country composed ...,rural-urban migration;indigenous migration;agr...,331
289,https://openalex.org/W7221013781,https://doi.org/10.82444/warw.33845461,Foodscapes of bread in sixteenth-century New S...,32.660973,2022,2022-06-23,en,https://doi.org/10.82444/warw.33845461,https://openalex.org/S7407060609,NaN,...,2026-10-28T10:33:24.094632,https://openalex.org/A5060377380;https://opena...,Ricardo Aguilar González;Godwin Yeboah,,,,,In this paper we analyse the introduction of w...,wheat bread;bread;wheat cultivation;colonialis...,120
990,https://openalex.org/W3196395180,https://doi.org/10.18356/9789210041065c045,No. 48030. International Development Associati...,199.181120,2020,2020-12-29,en,https://doi.org/10.18356/9789210041065c045,https://openalex.org/S4210238570,United Nations,...,2026-10-10T09:11:00.788812,,,,,,,FINANCING AGREEMENT (SOCIAL PRO-TECTION PROJEC...,social protection;financing agreement;Internat...,123


## Filtrar por topics/subfield
A diferencia del filtro por topics del documento usaré la clasificación por subfield debido a que es más fácil de estandarizar usando el archivo subfields_tagged.xlsx creado cruzando los estudios usados en el documento de referencia.

Una primera versión de `subfields_tagged_V1.xlsx` se construyó filtrando los 252 subcampos de OpenAlex con aquellos que están representados por los estudios usados en el reporte bibliométrico desarrollado manualmente.

La segunda versión, `subfields_tagged.xlsx` usado, define los topicos de interés mediante la revisión de las descripciones de cada subcampo por medio del chat de Claude con el modelo Opus 5.5.

In [19]:
import pandas as pd

subfields_df = pd.read_excel("subfields_tagged.xlsx")

In [20]:
subfields_df['interest_subfield'].value_counts()

interest_subfield
0    166
1     86
Name: count, dtype: int64

In [21]:
# Join ids
studies_df['primary_topic.subfield.id'] = studies_df['primary_topic.subfield.id'].str.split("/").str[-1].astype(str)
subfields_df = subfields_df.rename(columns={"id": "primary_topic.subfield.id"})
subfields_df['primary_topic.subfield.id'] = subfields_df['primary_topic.subfield.id'].astype(str)

In [22]:
studies_df = pd.merge(studies_df, subfields_df[['primary_topic.subfield.id', 'interest_subfield']], 
                       how="left", on ="primary_topic.subfield.id")

In [23]:
studies_df['interest_subfield'].value_counts(dropna=False)

interest_subfield
1.0    1451
0.0     325
NaN       9
Name: count, dtype: int64

In [24]:
studies_df.query("interest_subfield.isna()")[['display_name', 'primary_topic.subfield.id', 'primary_topic.subfield.display_name']]

,display_name,primary_topic.subfield.id,primary_topic.subfield.display_name
283,Publicaciones 2025 de la UNAH,NaN,NaN
830,[4K-HD] @Verpelis! Ava (2020) Pelicula Complet...,NaN,NaN
980,Número Completo (octubre-diciembre),NaN,NaN
984,Número completo (Enero-marzo),NaN,NaN
1705,Número Completo Volumen 29 Número 1,NaN,NaN
1734,{ +27788889342 } Bring Back Lost Love Spells I...,NaN,NaN
1755,psychic readings +27788889342 lost love spell ...,NaN,NaN
1769,Número Completo (octubre-diciembre),NaN,NaN
1781,After. En mil pedazos P E L I C U L A Completa...,NaN,NaN


Se descartan artículos sin subfield dado que son errores que el modelo de OA no pudo etiquetar.

In [25]:
studies_df['type'].value_counts(dropna=False)

type
article                1260
book-chapter            138
conference-paper        101
report                   98
book                     88
dissertation             84
conference-abstract      16
Name: count, dtype: int64

Conservar solo estudios del subfield de interés

In [26]:
studies_df.query("interest_subfield == 1", inplace=True)
studies_df.drop(columns=["interest_subfield"], inplace=True)

In [27]:
studies_df.shape

(1451, 40)

## Manejar duplicados

In [28]:
print("ID duplicados:", studies_df['id'].duplicated().sum())
print("Doi duplicados:", studies_df['doi'].dropna().duplicated().sum())
print("Nombres duplicados:", studies_df['display_name'].dropna().duplicated().sum())

ID duplicados: 176
Doi duplicados: 145
Nombres duplicados: 214


In [29]:
print("ID vacíos", len(studies_df[studies_df['id'].isna()]))
print("DOI vacíos", len(studies_df[studies_df['doi'].isna()]))
print("Nombres vacíos", len(studies_df[studies_df['display_name'].isna()]))

ID vacíos 0
DOI vacíos 239
Nombres vacíos 0


In [30]:
before_dedupl = studies_df.copy()

In [31]:
def check(title):
    clean = title.str.casefold().str.replace(r'\s+', ' ', regex=True).str.strip()
    return clean.str.split(' ').str.len()
before_dedupl['title_word_count'] = check(before_dedupl['display_name'])
    

In [32]:
before_dedupl.drop_duplicates(subset=['id'], keep='first', inplace=True)

In [33]:
before_dedupl[before_dedupl['display_name'].duplicated()][['doi', 'display_name', 'title_word_count']].\
sort_values(by='title_word_count', ascending=False)

,doi,display_name,title_word_count
1322,NaN,"Reducción del costo de importación, desde Hond...",23
154,NaN,Integration of indigenous people into sustaina...,22
372,NaN,Statebuilding and indigenous rights implementa...,16
735,https://doi.org/10.15665/rp.v18i2.2202,Eficiencia del gasto público y privado en salu...,15
1028,https://doi.org/10.6084/m9.figshare.30828891,Economic empowerment through welfare? MAPPING ...,13
1257,https://doi.org/10.1596/1813-9450-10826,Understanding Vulnerability to Poverty and Nat...,13
1103,NaN,Unraveling the Paradox of Anticorruption Messa...,13
864,https://doi.org/10.51952/9781529231748.ch004,Deconcentration of Education in Honduras: Rest...,12
861,https://doi.org/10.56687/9781529231748-008,Deconcentration of Education in Honduras: Rest...,12
1139,https://doi.org/10.33899/tanra.2022.176210.1185,Does The Distribution of Income Respond to the...,11


Identificación de duplicados en cascada: `id` -> `doi` -> título normalizado (`display_name`). Las llaves vacías se ignoran en cada paso. El título se normaliza (minúsculas y espacios) y solo se usa si tiene 4 palabras o más, para no fusionar títulos genéricos como "Honduras" o "Introduction".

En cada grupo se conserva la fila con mayor `relevance_score`, excepto en `thematic_id`:
- El `strategic_objective` es el de la temática con mayor `relevance_score` entre las filas fusionadas.
- Se conservan todos los `thematic_id` de ese `strategic_objective`, unidos con `;`.

In [34]:
def title_key(titles, min_words=4):
    ''' Normalized title as duplicate key, NaN for generic titles with less than min_words '''
    titles = titles.str.casefold().str.replace(r'\s+', ' ', regex=True).str.strip()
    return titles.where(titles.str.split(' ').str.len() >= min_words)

def dedup_step(df, key):
    ''' Merge rows sharing a non-null key: keep the row with highest relevance_score
    and collect the thematic matches of all merged rows '''
    df = df.sort_values('relevance_score', ascending=False, kind='stable')
    has_key = df[key].notna()
    matches = df[has_key].groupby(key, sort=False)['thematic_matches'].agg(
        lambda match_lists: [match for match_list in match_lists for match in match_list]
        )
    df_dedup = df[has_key].drop_duplicates(key, keep='first').copy()
    df_dedup['thematic_matches'] = df_dedup[key].map(matches)
    return pd.concat([df_dedup, df[~has_key]])

def assign_thematic_id(matches, so_by_thematic):
    ''' Keep the strategic objective of the most relevant match and all its thematic_ids '''
    top_thematic = max(matches, key=lambda match: match[1])[0]
    top_so = so_by_thematic[top_thematic]
    return ';'.join(sorted({thematic for thematic, _ in matches if so_by_thematic[thematic] == top_so}))

In [35]:
so_by_thematic = thematics_df.set_index('thematic_id')['strategic_objective']

studies_dedup = studies_df.copy()
studies_dedup['thematic_matches'] = [[(thematic_id, relevance)] for thematic_id, relevance
                                    in zip(studies_dedup['thematic_id'], studies_dedup['relevance_score'])]
studies_dedup['title_key'] = title_key(studies_dedup['display_name'])

print('Rows before:', len(studies_dedup))
for key in ['id', 'doi', 'title_key']:
    n_rows = len(studies_dedup)
    studies_dedup = dedup_step(studies_dedup, key)
    print(f'Duplicates removed by {key}:', n_rows - len(studies_dedup))

studies_dedup['thematic_id'] = studies_dedup['thematic_matches'].apply(
    assign_thematic_id, so_by_thematic=so_by_thematic)
studies_df = studies_dedup.drop(columns=['thematic_matches', 'title_key']).reset_index(drop=True)
print('Rows after:', len(studies_df))

Rows before: 1451


Duplicates removed by id: 176
Duplicates removed by doi: 1
Duplicates removed by title_key: 14
Rows after: 1260


In [36]:
print("ID duplicados:", studies_df['id'].duplicated().sum())
print("DOI duplicados (sin vacíos):", studies_df['doi'].dropna().duplicated().sum())
print("Títulos duplicados (≥ 4 palabras):", title_key(studies_df['display_name']).dropna().duplicated().sum())
print("Títulos duplicados:", studies_df['display_name'].dropna().duplicated().sum())
print("Estudios con múltiples temáticas:", studies_df['thematic_id'].str.contains(';').sum())

ID duplicados: 0
DOI duplicados (sin vacíos): 0
Títulos duplicados (≥ 4 palabras): 0
Títulos duplicados: 24
Estudios con múltiples temáticas: 45


Estudios con múltiples ejes temáticos

In [37]:
studies_df[studies_df['thematic_id'].str.contains(';')][['display_name', 'relevance_score', 'thematic_id']].head(10)

,display_name,relevance_score,thematic_id
4,Sostenibilidad fiscal en Honduras,230.38725,210;211
17,Burden of serious fungal infections in Honduras,206.82796,120;121
72,Political history and safe motherhood disparit...,177.13824,120;121
82,Test Empírico de Dominancia Fiscal en Honduras...,175.05437,210;212
117,"La educación inclusiva en Honduras: avances, o...",166.84381,120;122
130,Acceso al financiamiento agrícola en caficulto...,163.74683,321;322
136,Reforma fiscal en Honduras: lineamientos de po...,163.12558,210;211
156,Una Aproximación Teórica sobre la Educación In...,159.44717,120;122
232,Geographic Variation in Cardiometabolic and Nu...,148.34416,120;121
283,Schools caught in conflict: lessons from Hondu...,141.26375,121;122


In [38]:
studies_df.doi.isna().sum()

np.int64(201)

In [39]:
del studies_dedup

In [40]:
studies_df.columns

Index(['id', 'doi', 'display_name', 'relevance_score', 'publication_year',
       'publication_date', 'language', 'primary_location.landing_page_url',
       'primary_location.source.id',
       'primary_location.source.host_organization_name',
       'primary_location.source.display_name',
       'primary_location.raw_source_name', 'primary_location.source.type',
       'primary_location.raw_type', 'type', 'primary_location.is_oa',
       'open_access.is_oa', 'open_access.oa_status', 'open_access.oa_url',
       'fwci', 'cited_by_count', 'citation_normalized_percentile.value',
       'primary_topic.id', 'primary_topic.display_name',
       'primary_topic.subfield.id', 'primary_topic.subfield.display_name',
       'primary_topic.field.id', 'primary_topic.field.display_name',
       'primary_topic.domain.id', 'primary_topic.domain.display_name',
       'updated_date', 'authorships.author.id',
       'authorships.author.display_name', 'authorships.institutions.id',
       'authorships.in

## Imputar valores

Hay que indagar si buscando en OpenAlex por author id se puede obtener más información de las instituciones.

In [41]:
studies_df.isna().sum().to_frame(name='missing_count').\
    query("missing_count > 0").sort_values(by='missing_count', ascending=False)

,missing_count
primary_location.source.host_organization_name,466
open_access.oa_url,338
primary_location.raw_source_name,208
doi,201
fwci,158
citation_normalized_percentile.value,158
primary_location.source.id,120
primary_location.source.display_name,120
primary_location.source.type,120
abstract,110


In [42]:
(studies_df.map(lambda x: x.lower().strip() if isinstance(x, str) else x) == "").\
    sum().to_frame(name='empty_count').\
        query("empty_count > 0").sort_values(by='empty_count', ascending=False)

,empty_count
authorships.institutions.country_code,517
authorships.institutions.display_name,517
authorships.institutions.id,517
authorships.institutions.type,517
authorships.author.id,92
authorships.author.display_name,28
primary_location.raw_source_name,20


Preparación: textos vacíos como NaN y respaldos de OpenAlex indexados por `id` (para consultar las `locations` secundarias), excluyendo info de autores, se trabajará más adelante.

In [43]:
text_columns = ['doi', 'language', 'abstract', 'primary_location.source.host_organization_name',
                'primary_location.source.display_name', 'primary_location.raw_source_name']
studies_df[text_columns] = studies_df[text_columns].apply(lambda column: column.where(column.str.strip() != ''))

DOI

In [44]:
no_doi = studies_df['doi'].isna().rename('no_doi')
print("Studies without DOI and a DOI pattern in landing_page_url:",
      studies_df.loc[no_doi, 'primary_location.landing_page_url'].str.contains(r'10\.\d{4,9}/', regex=True).sum())

doi_by_type = pd.crosstab(studies_df['type'], no_doi, margins=True)
doi_by_type['pct_no_doi'] = (doi_by_type[True] / doi_by_type['All'] * 100).round(1)
doi_by_type.sort_values('pct_no_doi', ascending=False)

Studies without DOI and a DOI pattern in landing_page_url: 0


no_doi,False,True,All,pct_no_doi
type,,,,
dissertation,12,54,66,81.8
book,47,20,67,29.9
report,57,23,80,28.7
All,1059,201,1260,16.0
article,755,101,856,11.8
conference-paper,73,2,75,2.7
book-chapter,103,1,104,1.0
conference-abstract,12,0,12,0.0


No es posible imputar el DOI desde `primary_location.landing_page_url`: ninguna URL de los estudios sin DOI contiene un patrón `10.xxxx/` (son handles, repositorios institucionales, Dialnet, documents.worldbank.org, etc.) y en los respaldos tampoco hay DOI en las demás `locations` ni en `ids`.

Los faltantes se concentran en `dissertation`, `report` y `book` alojados en repositorios: literatura gris que normalmente no tiene DOI registrado. Se dejan como NaN.

Abstract

Se busca por DOI en Semantic Scholar y luego en Europe PMC (Crossref casi no aporta porque OpenAlex ya lo incluye). Se imputa antes que el idioma para que la detección pueda usar los abstracts recuperados.

In [45]:
from html import unescape

# Semantic Scholar API key
S2_HEADERS = {"x-api-key": os.getenv("S2_API_KEY")} if os.getenv("S2_API_KEY") else {} 

def fetch_s2_abstracts(dois, retries=10):
    ''' Abstracts from Semantic Scholar batch endpoint by DOI (max 100 ids per request) '''
    abstracts = {}
    for start in range(0, len(dois), 100):
        batch = dois[start:start + 100]
        for attempt in range(retries):
            req = requests.post("https://api.semanticscholar.org/graph/v1/paper/batch", 
                                params={"fields": "abstract"},
                                json={"ids": [f"DOI:{doi}" for doi in batch]}, 
                                headers=S2_HEADERS)
            if req.status_code != 429:
                break
            time.sleep(5)  # Shared rate limit without API key
        if req.status_code != 200:
            print(f"Semantic Scholar error: {req.status_code}")
            continue
        abstracts.update({doi: paper['abstract'] for doi, paper in zip(batch, req.json())
                        if paper and paper.get('abstract')})
    return abstracts

def fetch_epmc_abstract(doi):
    ''' Abstract from Europe PMC by DOI, without HTML tags (None if not found) '''
    req = requests.get("https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                        params={"query": f'DOI:"{doi}"', "resultType": "core", "format": "json"})
    if req.status_code != 200:
        return None
    results = req.json().get('resultList', {}).get('result', [])
    if not results or not results[0].get('abstractText'):
        return None
    return re.sub(r'\s+', ' ', unescape(re.sub(r'<[^>]+>', ' ', results[0]['abstractText']))).strip()

In [46]:
studies_df['abstract_source'] = pd.Series('openalex', index=studies_df.index).\
    where(studies_df['abstract'].notna())

missing_abstract = studies_df['abstract'].isna() & studies_df['doi'].notna()
dois = studies_df.loc[missing_abstract, 'doi'].str.removeprefix('https://doi.org/')

s2_abstracts = dois.map(fetch_s2_abstracts(dois.tolist())).dropna()
studies_df.loc[s2_abstracts.index, 'abstract'] = s2_abstracts
studies_df.loc[s2_abstracts.index, 'abstract_source'] = 'semantic_scholar'

epmc_abstracts = dois.drop(s2_abstracts.index).map(fetch_epmc_abstract).dropna()
studies_df.loc[epmc_abstracts.index, 'abstract'] = epmc_abstracts
studies_df.loc[epmc_abstracts.index, 'abstract_source'] = 'europepmc'

studies_df['abstract_source'].value_counts(dropna=False)

Semantic Scholar error: 429


abstract_source
openalex     1150
NaN            99
europepmc      11
Name: count, dtype: int64

In [47]:
studies_df.query("abstract_source == 'semantic_scholar'")[['doi', 'abstract']].head(5)

,doi,abstract


Idioma

En los respaldos no hay otro campo de idioma, así que se detecta con `lingua`. En los estudios que sí tienen idioma, la detección sobre el título coincide con OpenAlex en ~93 % y sobre el abstract en ~84 %: muchos estudios en español traen el abstract traducido al inglés. Por eso decide el título y el abstract solo se usa si el título no permite detectar el idioma.

In [48]:
studies_df['language'].value_counts(dropna=False)

language
en    694
es    537
pt     15
fr      6
id      2
ko      1
cs      1
gl      1
tr      1
it      1
sv      1
Name: count, dtype: int64

In [49]:
from lingua import Language, LanguageDetectorBuilder

language_detector = LanguageDetectorBuilder.from_languages(
    Language.ENGLISH, Language.SPANISH, Language.PORTUGUESE, Language.FRENCH, Language.ITALIAN, Language.GERMAN).build()

def detect_language(*texts):
    ''' ISO 639-1 code of the first text with a detectable language (None if none) '''
    for text in texts:
        if isinstance(text, str) and text.strip():
            language = language_detector.detect_language_of(text)
            if language is not None:
                return language.iso_code_639_1.name.lower()
    return None

In [50]:
# First try with display_name, second with abstract if display_name is empty
studies_df['language_imputed'] = studies_df['language'].isna()
studies_df.loc[studies_df['language_imputed'], 'language'] = studies_df[studies_df['language_imputed']].apply(
    lambda row: detect_language(row['display_name'], row['abstract']), axis=1)

print("Studies without language:", studies_df['language'].isna().sum())
studies_df.loc[studies_df['language_imputed'], 'language'].value_counts()

Studies without language: 0


Series([], Name: count, dtype: int64)

In [51]:
studies_df['language'].value_counts(dropna=False)

language
en    694
es    537
pt     15
fr      6
id      2
ko      1
cs      1
gl      1
tr      1
it      1
sv      1
Name: count, dtype: int64

Tipos

In [52]:
studies_df['type'].value_counts(dropna=False)

type
article                856
book-chapter           104
report                  80
conference-paper        75
book                    67
dissertation            66
conference-abstract     12
Name: count, dtype: int64

In [53]:
studies_df['primary_location.source.type'].value_counts(dropna=False)

primary_location.source.type
journal           781
repository        166
NaN               120
ebook platform    112
conference         60
book series        21
Name: count, dtype: int64

Imputar y unificar la organización que publica el estudio.

In [54]:
location_columns = [column for column in studies_df.columns if column.startswith('primary_location')]
studies_df[location_columns].sample(5, random_state=41)

,primary_location.landing_page_url,primary_location.source.id,primary_location.source.host_organization_name,primary_location.source.display_name,primary_location.raw_source_name,primary_location.source.type,primary_location.raw_type,primary_location.is_oa
149,https://doi.org/10.5377/eya.v17i2.21504,https://openalex.org/S4210189016,NaN,Economía y Administración (E&A),Economía y Administración (E&amp;A),journal,journal-article,True
756,https://doi.org/10.1007/s11625-021-00922-1,https://openalex.org/S135086837,Springer Science+Business Media,Sustainability Science,Sustainability Science,journal,journal-article,False
649,https://hdl.handle.net/11362/45299,https://openalex.org/S4377196404,United Nations,DIGITAL REPOSITORY Economic Commission for Lat...,NaN,repository,libro,False
688,https://figshare.com/articles/thesis/_b_Drug_t...,https://openalex.org/S4377196282,Figshare (United Kingdom),Figshare,NaN,repository,Thesis,True
635,http://repositorio.unan.edu.ni/10787/2/cc.jpg,https://openalex.org/S4306400730,National Autonomous University of Nicaragua,Repositorio Universidad Nacional Autonoma de N...,NaN,repository,Thesis,False


Cascada para `publisher` (editorial primero): `host_organization_name` → `display_name` → `raw_source_name` → editorial registrada para el DOI (Crossref o DataCite vía doi.org) → dominio de `landing_page_url`. `publisher_source` guarda el paso que aportó el valor.

In [55]:
DOMAIN_PUBLISHERS = {
    "documents.worldbank.org": "World Bank",
    "pesquisa.bvsalud.org": "Biblioteca Virtual en Salud (BVS)",
    "researchgate.net": "ResearchGate",
}
DOI_HEADERS = {"Accept": "application/vnd.citationstyles.csl+json", 
                "User-Agent": "Reporte-bibliometrico-BID/1.0"}

def clean_name(name):
    ''' Collapse whitespace; None for empty or non-text values '''
    if not isinstance(name, str):
        return None
    return re.sub(r'\s+', ' ', name).strip() or None

def doi_publisher(doi):
    ''' Publisher registered for the DOI (Crossref or DataCite) via doi.org content negotiation '''
    try:
        req = requests.get(doi, headers=DOI_HEADERS, timeout=30)
        publisher = req.json().get('publisher') if req.status_code == 200 else None
    except (requests.RequestException, ValueError) as e:
        print(f"Warning: {doi} ({type(e).__name__})")
        return None
    if req.status_code != 200:
        print(f"Warning: {doi} ({req.status_code})")
    return publisher if publisher != "Unpublished" else None  # ResearchGate DOIs are registered as "Unpublished"

def landing_publisher(url):
    ''' Publisher from the landing page domain; doi.org links are resolved to their final domain '''
    if re.match(r'https?://(dx\.)?doi\.org/', url):
        try:
            url = requests.head(url, allow_redirects=True, timeout=30).url
        except requests.RequestException:
            pass
    domain = re.sub(r'^https?://(www\.)?', '', url).split('/')[0]
    return DOMAIN_PUBLISHERS.get(domain, domain)

In [56]:
publisher_steps = {
    'host_organization_name': lambda df: df['primary_location.source.host_organization_name'],
    'display_name': lambda df: df['primary_location.source.display_name'],
    'raw_source_name': lambda df: df['primary_location.raw_source_name'],
    'doi': lambda df: df['doi'].dropna().map(doi_publisher),
    'landing_page_url': lambda df: df['primary_location.landing_page_url'].map(landing_publisher),
}

studies_df['publisher'] = None
studies_df['publisher_source'] = None
for step, get_names in publisher_steps.items():
    names = get_names(studies_df[studies_df['publisher'].isna()]).map(clean_name).dropna()
    studies_df.loc[names.index, 'publisher'] = names
    studies_df.loc[names.index, 'publisher_source'] = step

studies_df['publisher'] = studies_df['publisher'].str.replace('eBooks', '').str.strip()


assert studies_df['publisher'].notna().all()
studies_df['publisher_source'].value_counts(dropna=False)

publisher_source
host_organization_name    794
display_name              346
doi                        49
landing_page_url           43
raw_source_name            28
Name: count, dtype: int64

In [57]:
sel_columns = ['publisher_source', 'publisher', 'primary_location.source.host_organization_name',
                'primary_location.source.display_name', 'primary_location.raw_source_name']
studies_df.loc[studies_df['publisher_source'] == 'landing_page_url', sel_columns].head(10)

,publisher_source,publisher,primary_location.source.host_organization_name,primary_location.source.display_name,primary_location.raw_source_name
6,landing_page_url,World Bank,NaN,NaN,NaN
61,landing_page_url,repositorio.grial.eu,NaN,NaN,NaN
64,landing_page_url,opus.uni-hohenheim.de,NaN,NaN,NaN
89,landing_page_url,Biblioteca Virtual en Salud (BVS),NaN,NaN,NaN
117,landing_page_url,conferences.epistemopolis.org,NaN,NaN,NaN
124,landing_page_url,icdf.org.tw,NaN,NaN,NaN
137,landing_page_url,dspace.unive.it,NaN,NaN,NaN
164,landing_page_url,World Bank,NaN,NaN,NaN
177,landing_page_url,World Bank,NaN,NaN,NaN
219,landing_page_url,ResearchGate,NaN,NaN,NaN


Open Access flag

In [58]:
pd.crosstab(studies_df['open_access.oa_status'], studies_df['primary_location.is_oa'])

primary_location.is_oa,False,True
open_access.oa_status,,
bronze,0,99
closed,338,0
diamond,2,382
gold,0,124
green,60,128
hybrid,0,127


In [59]:
pd.crosstab(studies_df['doi'].isna(), studies_df['abstract'].isna())

abstract,False,True
doi,,
False,982,77
True,179,22


FWCI y citaciones

In [60]:
pd.crosstab(studies_df['fwci'].isna(), studies_df['citation_normalized_percentile.value'].isna())

citation_normalized_percentile.value,False,True
fwci,,
False,1102,0
True,0,158


In [61]:
studies_df[studies_df['fwci'].isna()].cited_by_count.value_counts(dropna=False)

cited_by_count
0     137
1       9
2       3
5       2
3       2
4       2
45      1
16      1
24      1
Name: count, dtype: int64

In [62]:
pd.crosstab(studies_df['type'], studies_df['fwci'].isna().rename('fwci_null'))

fwci_null,False,True
type,,
article,856,0
book,67,0
book-chapter,104,0
conference-abstract,0,12
conference-paper,75,0
dissertation,0,66
report,0,80


`fwci` y `citation_normalized_percentile.value` **no se imputan con 0**:

- FWCI = citas recibidas (año de publicación + 3 años) / citas esperadas (promedio de trabajos del mismo año, tipo y subcampo); 1.0 es el promedio mundial ([OpenAlex](https://help.openalex.org/data/works/citations/)). El percentil expresa lo mismo en escala percentil.
- OpenAlex omite el FWCI en los tipos que "no se espera que acumulen citas". En este conjunto los nulos son exactamente `report`, `dissertation` y `conference-abstract` (ninguno tiene FWCI) y algunos tienen 16, 24 o 45 citas: el nulo significa "no aplica", no "cero impacto".
- Ni siquiera un estudio sin citas tiene percentil 0 (por los empates, su percentil varía, con mediana ~0.27), así que 0 sería un valor inventado.

En el análisis, los promedios de FWCI y percentil se calculan solo sobre los tipos elegibles.

#### Autores e instituciones

Se tratan como entidades independientes

Las columnas `authorships.*` se separan por ";", se imputan como listas y se vuelven a unir. **Criterio de alineación:** en cada estudio, todas las columnas de un grupo tienen el mismo número de elementos:
- autores: `authorships.author.id`, `.display_name` e `.id_imputed`;
- instituciones: `authorships.institutions.id`, `.display_name`, `.country_code`, `.type` y `.root_display_name`.

Cuando un campo tiene varios valores y falta uno de ellos (p. ej. un autor sin id entre autores que sí lo tienen), ese elemento se reemplaza por `MISSING` ("Not indicated") para conservar la posición. Es distinto de un campo vacío por completo: un estudio sin autores o sin instituciones queda NaN en todo el grupo.

- **Estudios sin authorships**: ni el DOI (Crossref/DataCite) ni la página de aterrizaje recuperan autores de forma útil. Muchos son documentos con autoría institucional (Naciones Unidas, OPS): si la fuente no es una revista y el publisher coincide con una institución de OpenAlex que no es empresa ni universidad, se usa como autor (`publisher_as_author`). Los demás (Wiley, Lexington Books, revistas…) quedan NaN.
- **Autores que son instituciones** (World Bank, OECD, BID…), solo en estudios sin instituciones: se buscan en `/institutions` y se asignan si la similitud del nombre con el nombre, las alternativas o las siglas de la institución es ≥ 0.85.
- **Autores sin id** (solo en estudios sin instituciones): se buscan en `/authors`. Se asigna el id si el nombre normalizado coincide y el autor tiene una afiliación a ±1 año de la publicación, siempre que haya un único candidato válido o que todos compartan una institución (perfiles duplicados del mismo autor). En una prueba con ids conocidos, esta regla acertó el 95 % y recupera ~30 % de los autores sin id.
- **Estudios sin instituciones:** se toman las afiliaciones de sus autores (con id) en el mismo año de publicación. No se usan las `raw_affiliation_strings`, porque dan falsos positivos ("ORCiD" → ORCID).

In [63]:
import difflib
import unicodedata
from collections import Counter

MISSING = "Not indicated"
AUTHOR_COLS = ['authorships.author.id', 'authorships.author.display_name', 'authorships.author.id_imputed']
INST_COLS = [f'authorships.institutions.{inst_var}' for inst_var in institutions_vars] + ['authorships.institutions.root_display_name']
INST_SELECT = "id,display_name,display_name_alternatives,display_name_acronyms,country_code,type" # Select fields for institution search in OpenAlex
INST_TREE_SELECT = "id,display_name,ror,country_code,type,associated_institutions,works_count" # Select fields for institution tree in OpenAlex
AUTHORS_SELECT = "id,display_name,display_name_alternatives,affiliations,relevance_score" # Select fields for author search in OpenAlex 
INSTITUTION_AUTHOR_TYPES = {'government', 'nonprofit', 'other', 'healthcare', 'facility', 'funder'} # Institution types accepted as institutional authors
oa_cost = {'usd': 0}
ror_cache = {} # Research Organization Registry ROR

short_id = lambda openalex_id: openalex_id.rsplit('/', 1)[-1]

def split_list(value):
    ''' ";" joined text as list ([] for NaN or empty text) '''
    return value.split(';') if isinstance(value, str) and value.strip() else []

def join_list(values):
    ''' List as ";" joined text, MISSING for empty elements (None for empty lists) '''
    if not values:
        return None
    return ';'.join(MISSING if value is None or value == '' else str(value).replace(';', ',') for value in values)

def oa_get(path, params, retries=3):
    ''' GET to the OpenAlex API with retries, adding the cost to oa_cost'''
    for attempt in range(retries):
        try:
            req = requests.get(OA_BASE + path, params=params, headers=OA_HEADERS, timeout=60)
            if req.status_code == 200:
                data = req.json()
                oa_cost['usd'] += data['meta'].get('cost_usd') or 0
                return data
        except (requests.RequestException, ValueError):
            pass
        time.sleep(2 * (attempt + 1))
    print(f"Warning: {path} failed ({params}) with code {req.status_code if 'req' in locals() else 'unknown'})")
    return None

def norm_name(text):
    ''' Name without accents, case, punctuation or extra spaces '''
    text = unicodedata.normalize('NFKD', text or '').encode('ascii', 'ignore').decode().casefold()
    return ' '.join(re.sub(r'[.,\-‐]', ' ', text).split())

def name_similarity(name, candidates):
    ''' Highest similarity between the normalized name and the candidates '''
    similarities = (difflib.SequenceMatcher(None, norm_name(name), norm_name(candidate)).ratio()
                for candidate in candidates if candidate)
    return max(similarities, default=0)

def fetch_entities(entity, ids, select):
    ''' OpenAlex entities by id in batches of 50 ({id: entity}) '''
    ids = sorted({short_id(openalex_id) for openalex_id in ids if openalex_id and openalex_id != MISSING})
    entities = {}
    for start in range(0, len(ids), 50):
        data = oa_get(entity, {"filter": "ids.openalex:" + "|".join(ids[start:start + 50]),
                               "per_page": 50, "select": select})
        entities.update({result['id']: result for result in (data or {}).get('results', [])})
    return entities

def inst_record(inst):
    ''' Institution fields used in studies_df '''
    return {inst_var: inst.get(inst_var) for inst_var in institutions_vars}

def match_institution(name, threshold=0.85):
    ''' OpenAlex institution matching a name (full name, then its first segment with 2+ words).
    The most similar of the first 10 results is kept: the exact match is not always ranked first
    (e.g. "United Nations" comes after FAO) and the cost of a search does not depend on per_page '''
    first_segment = re.split(r'[.,(]', name)[0].strip()
    queries = [name] + ([first_segment] if first_segment != name and len(first_segment.split()) >= 2 else [])
    for query in queries:
        data = oa_get("institutions", {"search": query, "per_page": 10, "select": INST_SELECT})
        scored = [(name_similarity(query, [inst['display_name']] + (inst.get('display_name_alternatives') or [])
                                   + (inst.get('display_name_acronyms') or [])), inst)
                  for inst in (data or {}).get('results', [])]
        if scored:
            similarity, inst = max(scored, key=lambda item: item[0])
            if similarity >= threshold:
                return {**inst_record(inst), 'similarity': round(similarity, 2)}
    return None

def affiliation_ids(author, years):
    ''' Institution ids of the author affiliations in any of the years '''
    return {aff['institution']['id'] for aff in author.get('affiliations') or []
            if set(aff.get('years') or []) & years}

def search_author(name, year):
    ''' Author by name: same normalized name and an affiliation within ±1 year of the publication.
    Assigned when there is one valid candidate or all of them share an institution (duplicated profiles).
    Returns (author or None, number of valid candidates) '''
    data = oa_get("authors", {"search": name, "per_page": 10, "select": AUTHORS_SELECT})
    years = {year - 1, year, year + 1}
    valid = []
    for author in (data or {}).get('results', []):
        author_names = [author['display_name']] + (author.get('display_name_alternatives') or [])
        same_name = norm_name(name) in {norm_name(author_name) for author_name in author_names}
        if same_name and affiliation_ids(author, years):
            valid.append(author)
    if not valid:
        return None, 0
    if len(valid) == 1 or set.intersection(*(affiliation_ids(author, years) for author in valid)):
        return max(valid, key=lambda author: author.get('relevance_score') or 0), len(valid)
    return None, len(valid)

def fetch_ror(ror):
    ''' Name, country and type of an organization from the ROR API v2 ({} without ROR) 
    Research Organization Registry (ROR)'''
    if not ror:
        return {}
    if ror not in ror_cache:
        try:
            req = requests.get("https://api.ror.org/v2/organizations/" + short_id(ror), timeout=30)
            org = req.json() if req.status_code == 200 else {}
        except (requests.RequestException, ValueError):
            org = {}
        names = [name['value'] for name in org.get('names', []) if 'ror_display' in name.get('types', [])]
        location = (org.get('locations') or [{}])[0]
        ror_cache[ror] = {'display_name': names[0] if names else None,
                          'country_code': (location.get('geonames_details') or {}).get('country_code'),
                          'type': (org.get('types') or [None])[0]}
    return ror_cache[ror]

def fetch_institution_tree(ids):
    ''' OpenAlex entities of the institutions and all their parents, grandparents, etc.
    ids: iterable of OpenAlex institution ids (e.g. "https://openalex.org/I169009895").
    Returns {institution id: entity} with the fields of INST_TREE_SELECT. Used to fill missing
    fields (display_name, country_code, type, ror) and as input of root_institution() '''
    institutions, requested, pending = {}, set(), set(ids)
    while pending: # Iteratively fetch parents (.. grand parents) until no new ones are found
        requested = requested | pending
        institutions.update(fetch_entities('institutions', pending, INST_TREE_SELECT))
        pending = {parent['id'] for inst in institutions.values() 
                   for parent in inst.get('associated_institutions') or []
                   if parent.get('relationship') == 'parent'} - requested
    return institutions

def root_institution(inst_id, institutions):
    ''' Id of the root parent institution of inst_id.
    inst_id: OpenAlex institution id
    institutions: {institution id: entity} returned by fetch_institution_tree(); it must include
    every parent, with "associated_institutions", "type" and "works_count".
    Goes up the "parent" relations until an institution without parents. With several parents it
    picks the one with the same type, then the one with most works. Without parents returns inst_id '''
    visited = {inst_id}
    for _ in range(len(institutions) + 1): 
        inst = institutions.get(inst_id) or {}
        parents = [parent for parent in inst.get('associated_institutions') or []
                   if parent.get('relationship') == 'parent' and parent['id'] not in visited]
        if not parents:
            return inst_id
        inst_id = max(parents, key=lambda parent: (parent.get('type') == inst.get('type'),
                                                   (institutions.get(parent['id']) or {}).get('works_count') or 0))['id']
        visited.add(inst_id)
    return inst_id

def group_counts(df, columns):
    ''' Number of elements of each column of a group, per study '''
    return pd.concat([df[column].map(lambda value: len(split_list(value))) for column in columns], axis=1)

Separar en listas y alinear los autores. Los ids faltantes pasan a `MISSING`; si un nombre contiene ";" y rompe la alineación, los nombres se reconstruyen con las entidades de OpenAlex.
Costo OA API: $0.005.

In [64]:
# AUTHOR_COLS + INST_COLS Also include columns so create
# If  studies_df[column].fillna('').astype(str).str.strip() == '' then set to NaN
for column in [column for column in AUTHOR_COLS + INST_COLS if column in studies_df.columns]:
    studies_df[column] = studies_df[column].where(
        studies_df[column].fillna('').astype(str).str.strip() != '')
    
print("Studies without authors (before):", studies_df['authorships.author.display_name'].isna().sum())
print("Studies without institutions (before):", studies_df['authorships.institutions.id'].isna().sum())

author_ids = studies_df['authorships.author.id'].map(split_list)
author_names = studies_df['authorships.author.display_name'].map(split_list)
author_entities = fetch_entities('authors', author_ids.explode().dropna(), "id,display_name,affiliations")
print("Author entities:", len(author_entities))

# A ";" inside a name breaks the alignment: names are rebuilt from the author entities
misaligned = (author_ids.str.len() > 0) & (author_ids.str.len() != author_names.str.len())
author_names = author_names.mask(misaligned, author_ids.map(
    lambda ids: [(author_entities.get(author_id) or {}).get('display_name') for author_id in ids]))
print("Studies with names rebuilt from OpenAlex:", misaligned.sum())

author_ids = pd.Series([[author_id or MISSING for author_id in ids] if ids else [MISSING] * len(names)
                        for ids, names in zip(author_ids, author_names)], index=studies_df.index)
author_names = pd.Series([[name or (author_entities.get(author_id) or {}).get('display_name') or MISSING
                           for author_id, name in zip(ids, names)]
                          for ids, names in zip(author_ids, author_names)], index=studies_df.index)
id_imputed = author_ids.map(lambda ids: [False] * len(ids))

def study_institutions(row):
    ''' Institutions of a study as dicts; fields not aligned with the ids are left empty '''
    ids = split_list(row['authorships.institutions.id'])
    fields = {inst_var: split_list(row[f'authorships.institutions.{inst_var}']) for inst_var in institutions_vars}
    fields = {inst_var: values if len(values) == len(ids) else [None] * len(ids) for inst_var, values in fields.items()}
    return [{inst_var: None if fields[inst_var][k] in ('', MISSING) else fields[inst_var][k] for inst_var in institutions_vars}
            for k in range(len(ids))]

inst_lists = studies_df.apply(study_institutions, axis=1)
inst_sources = inst_lists.map(lambda insts: ['openalex'] if insts else [])

print("Studies without institutions (after):", inst_lists.map(len).eq(0).sum())
print("Authors without id:", sum(author_id == MISSING for ids in author_ids for author_id in ids))

Studies without authors (before): 28
Studies without institutions (before): 517
Author entities: 2813
Studies with names rebuilt from OpenAlex: 0
Studies without institutions (after): 517
Authors without id: 293


Autores que son instituciones: se busca como institución a los autores de los estudios **sin instituciones de OpenAlex** que no tienen id, o que tienen un perfil sin afiliaciones en estudios con ≤ 3 autores. La institución se agrega a todos los estudios donde aparece el nombre, si no estaba.

El filtro reduce las búsquedas de 690 a ~307 (costo OA API: $0.726 → ~$0.31). En la evaluación detectó 14 de las 19 instituciones (45 de 52 authorships). Las 5 que pierde están en estudios que ya tenían instituciones de OpenAlex.

In [65]:
# Only studies without OpenAlex institutions: authors without id, or authors without affiliations in studies with <= 3 authors
empty_studies = inst_lists[inst_lists.str.len() == 0].index
org_candidates = {name for index in empty_studies
                  for author_id, name in zip(author_ids[index], author_names[index])
                  if name != MISSING and (author_id == MISSING or 
                                          (len(author_names[index]) <= 3 and 
                                           not (author_entities.get(author_id) or {}).get('affiliations')))}
print("Candidates for institution matching:", len(org_candidates))
org_matches = {name: match for name in sorted(org_candidates) if (match := match_institution(name))} # Compare and assign the match if found
print("Matched institutions:", len(org_matches))

# Matched names are added in every study where they appear, keeping the order and avoiding duplicates
for index, names in author_names.items():
    for name in names:
        match = org_matches.get(name)
        if match and match['id'] not in {inst['id'] for inst in inst_lists[index]}:
            inst_lists[index].append(inst_record(match))
            if 'author_as_institution' not in inst_sources[index]:
                inst_sources[index].append('author_as_institution')

# How many studies have at least one institution matched from the authorship list
print("Studies with at least one institution matched from authorship:", 
        inst_sources.map(lambda sources: 'author_as_institution' in sources).sum())

Candidates for institution matching: 306
Matched institutions: 15
Studies with at least one institution matched from authorship: 44


In [66]:
pd.DataFrame.from_dict(org_matches, orient='index')[['display_name', 'country_code', 'type', 'similarity']]

,display_name,country_code,type,similarity
Banco Interamericano de Desarrollo,Inter-American Development Bank,US,other,1.00
"Catholic Relief Services (CRS), Nicaragua",Catholic Relief Services,US,nonprofit,1.00
Economic Commission for Latin America and the Caribbean,United Nations Economic Commission for Latin A...,CL,government,0.88
Inter American Development Bank,Inter-American Development Bank,US,other,1.00
Inter-American Development Bank,Inter-American Development Bank,US,other,1.00
International Finance Corporation,International Finance Corporation,KZ,government,1.00
International Monetary Fund. Fiscal Affairs Dept.,International Monetary Fund,US,other,1.00
International Monetary Fund. Western Hemisphere Dept.,International Monetary Fund,US,other,1.00
OECD,Organisation de Coopération et de Développemen...,FR,government,1.00
Rights and Resources Initiative,Rights and Resources,US,nonprofit,1.00


Publisher como autor: en los estudios sin ninguna authorship cuya fuente no es una revista, el publisher se asigna como autor institucional si coincide con una institución de OpenAlex (similitud ≥ 0.85) que no es empresa ni universidad. Las empresas son editoriales (Wiley, Springer…) y las universidades suelen alojar o editar documentos sin firmarlos (p. ej. un repositorio universitario). La institución también se agrega a las columnas de instituciones con la marca `publisher_as_author`.

Costo OA API: $0.01

In [67]:
no_authors = author_names[author_names.str.len() == 0].index
publisher_candidates = studies_df.loc[no_authors].query("`primary_location.source.type` != 'journal' and publisher.notna()")
print("Publisher candidates:", len(publisher_candidates))
publisher_matches = {publisher: match_institution(publisher) for publisher in publisher_candidates['publisher'].unique()}

# Add matched authors institutions
for index, publisher in publisher_candidates['publisher'].items():
    match = publisher_matches[publisher]
    if match and match['type'] in INSTITUTION_AUTHOR_TYPES:
        author_names[index].append(match['display_name'])
        author_ids[index].append(MISSING) # Not interested in ids
        id_imputed[index].append(False)
        inst_lists[index].append(inst_record(match))
        inst_sources[index].append('publisher_as_author')

print("Studies without authors:", len(no_authors), 
        "| Imputed with publisher:", (inst_sources.map(lambda s: 'publisher_as_author' in s)).sum())

Publisher candidates: 26
Studies without authors: 28 | Imputed with publisher: 24


Reasons for not used publisher

In [68]:
def publisher_decision(publisher):
    ''' Why a publisher was or was not used as institutional author '''
    match = publisher_matches.get(publisher)
    if publisher not in publisher_matches:
        return "journal"
    if match is None:
        return "no match"
    return "imputed" if match['type'] in INSTITUTION_AUTHOR_TYPES else f"excluded type ({match['type']})"

studies_df.loc[no_authors].groupby('publisher', dropna=False).size().rename('studies').reset_index().assign(
    institution=lambda df: df['publisher'].map(lambda p: (publisher_matches.get(p) or {}).get('display_name')),
    similarity=lambda df: df['publisher'].map(lambda p: (publisher_matches.get(p) or {}).get('similarity')),
    decision=lambda df: df['publisher'].map(publisher_decision))

,publisher,studies,institution,similarity,decision
0,Centro de Investigaciones y Capacitaciones Int...,1,NaN,NaN,no match
1,Erasmus University Rotterdam,1,Erasmus University Rotterdam,1.0,excluded type (education)
2,Organización Panamericana de la Salud,1,World Health Organization Regional Office for ...,1.0,imputed
3,Tanmiyaẗ al-rāfidayn,1,NaN,NaN,journal
4,United Nations,23,United Nations,1.0,imputed
5,Wiley,1,NaN,NaN,journal


Author id para los autores-persona sin id, solo en los estudios que siguen sin instituciones: el id imputado sirve para recuperar sus instituciones en el paso siguiente (~80 búsquedas en lugar de ~245). `authorships.author.id_imputed` marca las posiciones imputadas.
Costo OA API: $0.07

In [69]:
# Only studies still without institutions: the imputed ids are used to recover their institutions
empty_studies = inst_lists[inst_lists.str.len() == 0].index
author_queries = {(name, int(studies_df.at[index, 'publication_year']))
                  for index in empty_studies
                  for author_id, name in zip(author_ids[index], author_names[index])
                  if author_id == MISSING and name != MISSING and name not in org_matches}
author_matches = {query: search_author(*query) for query in sorted(author_queries)}

# Add ids to matched 
for index in empty_studies:
    ids, names = author_ids[index], author_names[index]
    year = int(studies_df.at[index, 'publication_year'])
    for k, (author_id, name) in enumerate(zip(ids, names)):
        author, _ = author_matches.get((name, year), (None, 0))
        if author_id == MISSING and author is not None:
            ids[k] = author['id']
            id_imputed[index][k] = True
            author_entities[author['id']] = author

print("Searched authors:", len(author_queries), 
      "| Imputed ids:", sum(author is not None for author, _ in author_matches.values()))

Searched authors: 71 | Imputed ids: 18


In [70]:
pd.DataFrame([{'name': name, 'year': year, 'id': author['id'], 'display_name': author['display_name'], 'n_valid': n_valid}
              for (name, year), (author, n_valid) in author_matches.items() if author is not None])

,name,year,id,display_name,n_valid
0,Abner Rodriguez,2026,https://openalex.org/A5102059853,Abner A. Rodríguez,1
1,Carolina Peters,2020,https://openalex.org/A5108856007,Carolina Peters,1
2,Catherine Koh,2023,https://openalex.org/A5050553355,Catherine Siow Lan Koh,1
3,Claudio Salgado,2019,https://openalex.org/A5018296587,Cláudio Guedes Salgado,1
4,Ginés Suarez,2020,https://openalex.org/A5070884067,Ginés Suárez,1
5,Giselle Del Carmen,2021,https://openalex.org/A5028031905,Giselle Del Carmen,1
6,Isabel Granada,2022,https://openalex.org/A5042537187,Isabel Granada,1
7,John Westfall,2025,https://openalex.org/A5074365032,John M. Westfall,1
8,Judith Alejandra Velázquez Castro,2020,https://openalex.org/A5036450273,Judith Alejandra Velázquez Castro,2
9,MARIA PINEDA,2021,https://openalex.org/A5103013782,María Pineda,1


Estudios sin instituciones: instituciones de las afiliaciones de sus autores en el año de publicación.

In [71]:
empty_studies = inst_lists[inst_lists.str.len() == 0].index
missing_entities = {author_id for index in empty_studies for author_id in author_ids[index]} - set(author_entities)
author_entities.update(fetch_entities('authors', missing_entities, "id,display_name,affiliations"))

for index in empty_studies:
    year = int(studies_df.at[index, 'publication_year'])
    institutions = {}
    for author_id in author_ids[index]:
        for aff in (author_entities.get(author_id) or {}).get('affiliations') or []:
            if year in (aff.get('years') or []):
                institutions.setdefault(aff['institution']['id'], inst_record(aff['institution'])) # Add id not exists
    if institutions:
        inst_lists[index].extend(institutions.values())
        inst_sources[index].append('author_affiliation')

print("Studies without institutions:", len(empty_studies), "->", (inst_lists.str.len() == 0).sum())

Studies without institutions: 449 -> 289


In [72]:
inst_sources.map(lambda sources: ';'.join(sources) or None).value_counts(dropna=False)

openalex                 743
NaN                      289
author_affiliation       160
author_as_institution     44
publisher_as_author       24
Name: count, dtype: int64

Estandarizar las instituciones:
1. Campos faltantes (`display_name`, `country_code`, `type`) en cascada: entidad de OpenAlex → ROR → `MISSING`.
2. Ids con el mismo nombre normalizado y el mismo país se unifican en el id más frecuente (p. ej. UNAH tiene dos ids).
3. `root_display_name`: organización padre raíz, subiendo por las relaciones `parent` de OpenAlex. Si hay varios padres, se elige el del mismo tipo y luego el de más trabajos.

In [73]:
institutions_parents = fetch_institution_tree({inst['id'] for insts in inst_lists for inst in insts})
print("Institution entities (with parents):", len(institutions_parents))

# Fill missing fields from institution tree
fill_counts = Counter()
for insts in inst_lists:
    for inst in insts:
        entity = institutions_parents.get(inst['id']) or {}
        for inst_var in ['display_name', 'country_code', 'type']:
            if inst.get(inst_var): # If have the value pass
                continue
            # If is missing try OpenAlex and ror
            for source, get_value in [('openalex', lambda: entity.get(inst_var)),
                                      ('ror', lambda: fetch_ror(entity.get('ror')).get(inst_var))]:
                if value := get_value(): # Compare and assign
                    inst[inst_var] = value
                    fill_counts[(inst_var, source)] += 1
                    break
print("Filled fields:", dict(fill_counts))

# Canonical id: same normalized name and country -> most frequent id
id_counts = Counter(inst['id'] for insts in inst_lists for inst in insts)
unique_insts = {inst['id']: inst for insts in inst_lists for inst in insts}
canonical, canonical_by_id = {}, {}
for inst_id, inst in sorted(unique_insts.items(), key=lambda item: -id_counts[item[0]]): # Ordered to use the most used id
    key = (norm_name(inst['display_name']), inst['country_code']) if inst['display_name'] else inst_id
    canonical_by_id[inst_id] = canonical.setdefault(key, inst)
merged = {inst_id: inst['id'] for inst_id, inst in canonical_by_id.items() if inst_id != inst['id']}
# merge = {original id : canonical id}
inst_lists = inst_lists.map(
    lambda insts: list({canonical_by_id[inst['id']]['id']: 
                        canonical_by_id[inst['id']] for inst in insts}.values())
    )

print("Count of merged ids:", len(merged))

# Create root names by study
root_names = {inst_id: 
              (institutions_parents.get(root_institution(inst_id, institutions_parents)) or {}).get('display_name') or inst['display_name']
              for inst_id, inst in unique_insts.items()}

Institution entities (with parents): 962
Filled fields: {}
Count of merged ids: 1


Reconstruir columnas en df

In [74]:
studies_df['authorships.author.id'] = author_ids.map(join_list)
studies_df['authorships.author.display_name'] = author_names.map(join_list)
studies_df['authorships.author.id_imputed'] = id_imputed.map(join_list)
for inst_var in institutions_vars:
    studies_df[f'authorships.institutions.{inst_var}'] = inst_lists.map(lambda insts: join_list([inst[inst_var] for inst in insts]))
studies_df['authorships.institutions.root_display_name'] = inst_lists.map(lambda insts: join_list([root_names[inst['id']] for inst in insts]))
studies_df['institutions_source'] = inst_sources.map(lambda sources: ';'.join(sources) or None)

# Checks
for group in (AUTHOR_COLS, INST_COLS):
    assert (group_counts(studies_df, group).nunique(axis=1) == 1).all(), f"Misaligned columns: {group}"
    assert not studies_df[group].map(lambda value: '' in split_list(value)).any().any(), f"Empty elements: {group}"

inst_names = pd.DataFrame([(inst['id'], inst['display_name']) for insts in inst_lists for inst in insts], 
                          columns=['id', 'display_name'])
assert (inst_names.groupby('id')['display_name'].nunique() == 1).all(), "Institution ids with more than one name"

print("Studies without authors / institutions (after):",
        studies_df['authorships.author.display_name'].isna().sum(), 
        "/", 
        studies_df['authorships.institutions.id'].isna().sum())

Studies without authors / institutions (after): 4 / 289


Muestra de instituciones origen

In [75]:
pd.DataFrame([{'institution': inst['display_name'], 'root': root_names[inst_id]}
                for inst_id, inst in unique_insts.items() 
                if root_names[inst_id] != inst['display_name']]).\
                drop_duplicates().sample(10)

,institution,root
169,United States Army Medical Directorate - Armed...,Government of the United States of America
70,Consorci Institut D'Investigacions Biomediques...,CERCA Institution
71,RTI International,Duke University
183,"California State University, Sacramento",California State University System
154,University of Arkansas for Medical Sciences,University of Arkansas System
48,Texas Tech University,Texas Tech University System
33,The Ohio State University,University System of Ohio
140,University of Georgia,University System of Georgia
22,United States Naval Medical Research Unit SOUTH,Government of the United States of America
166,Naval Medical Research Unit INDO PACIFIC,Government of the United States of America


In [76]:
print('Costo estimado OpenAlex API:', round(oa_cost['usd'], 3))

Costo estimado OpenAlex API: 0.402


### Columnas a eliminar
Columnas a eliminar, se hace aquí para conservar la razón, pero se eliminarán de interest_vars en pipeline.

- Eliminar las variable 'primary_location.raw_type' y 'primary_location.source.type ya que la variable 'type' cubre todo lo necesario. 
- Eliminar 'primary_location.is_oa' dado que tiene información redundando y no consistente con el status.
- 'primary_location.source.id' no es útil porque ya se trae su información disponible.
- 'primary_location.source.host_organization_name' y 'primary_location.source.display_name' cumplieron su rol creando publisher.
- 'authorships.author.id' cumple su propósito en las imputaciones y no se necesita más.

In [77]:
drop_cols = ['primary_location.source.type', 'primary_location.raw_type', 
            'primary_location.is_oa', 'primary_location.source.id',
            'primary_location.source.host_organization_name', 'primary_location.source.display_name',
            'authorships.author.id']
studies_df.drop(columns=drop_cols, inplace=True)

## Agrupar instituciones

Se clasifica cada institución en uno de los grupos de `INST_GROUPS`:
- `Honduras` (valor de `queries_honduras['meta']['country']`): instituciones del país.
- `LAC`: América Latina y el Caribe, excluyendo Honduras.
- `United States`: instituciones de Estados Unidos.
- `Europe`: instituciones de países europeos (incluye Reino Unido).
- `Multilateral organizations`: BID, Banco Mundial, OCDE, FMI, Naciones Unidas y sus agencias, CEPAL, OPS/OMS, BCIE, SICA y similares, sin importar su país.
- `Other institutions`: instituciones del resto del mundo.
- `Not indicated` (`MISSING`): no es posible determinar el origen con la información disponible.

Clasificación con Claude Haiku 5.5 (`claude-haiku-5-5`) con salida estructurada:
- Se clasifica cada **institución única** (por `authorships.institutions.id`), en lotes de 100, usando su nombre, `country_code`, `type` y organización padre raíz. Así una misma institución recibe siempre el mismo grupo en todos los estudios y se envían menos tokens que clasificando estudio por estudio.
- La nueva variable `authorships.institutions.inst_group` queda **alineada con las columnas de instituciones**: un grupo por institución, en el mismo orden y separado por `;`. Una institución que el modelo no clasifica queda como `MISSING`.
- Los estudios sin instituciones quedan con `inst_group` vacío (NaN), igual que el resto del grupo de instituciones. Los autores que son organizaciones ya se incorporaron como instituciones en "Autores e instituciones" (`author_as_institution` y `publisher_as_author`).

In [78]:
import anthropic
from typing import Literal
from pydantic import BaseModel

INST_GROUPS = (queries_honduras['meta']['country'], "LAC", "USA & Canada", 
                "Europe", "Multilateral organizations", "Other institutions", 
                MISSING)

class InstitutionGroup(BaseModel):
    key: int
    inst_group: Literal[INST_GROUPS]

class InstGroupBatch(BaseModel):
    institutions: list[InstitutionGroup]

INST_SYSTEM_PROMPT = f"""Clasifica el origen de las instituciones que participan en estudios académicos sobre {queries_honduras['meta']['country']} para un reporte bibliométrico.

Recibes una lista JSON de instituciones. Cada institución tiene un "key", su "name", "country_code" (ISO-2), "type" (tipo de OpenAlex) y "root" (organización padre raíz). Un valor null indica que el dato no está disponible.

Categorías:
- "{queries_honduras['meta']['country']}": instituciones de {queries_honduras['meta']['country']}.
- "LAC": instituciones de América Latina y el Caribe, excluyendo {queries_honduras['meta']['country']}.
- "USA & Canada": instituciones de Estados Unidos (US, USA) y Canadá.
- "Europe": instituciones de países europeos (incluye Reino Unido).
- "Multilateral organizations": BID/IDB, Banco Mundial, OCDE, FMI, Naciones Unidas y sus agencias (PNUD, UNICEF, FAO, OIT...), CEPAL, OPS/OMS, BCIE, SICA y similares, sin importar su country_code (por ejemplo, el Banco Mundial tiene country_code US pero es multilateral).
- "Other institutions": instituciones del resto del mundo.
- "{MISSING}": no es posible determinar el origen con la información disponible.

Reglas:
- Asigna exactamente una categoría a cada institución.
- Organismos multilaterales ("Multilateral organizations") tiene prioridad sobre el país. Usa también "root" para identificarlos (por ejemplo, una oficina nacional cuyo root es UNICEF).
- Si country_code es null, infiere el país a partir del nombre solo cuando sea evidente.
- Devuelve exactamente un resultado por cada key recibido."""

client = anthropic.Anthropic()

In [79]:
INST_CONTEXT_FIELDS = {'name': 'display_name', 'country_code': 'country_code', 'type': 'type', 'root': 'root_display_name'}

def build_inst_table(df):
    ''' Unique institutions of the studies (index: institution id) with the fields used in the classification '''
    columns = ['authorships.institutions.id'] + [f'authorships.institutions.{var}' for var in INST_CONTEXT_FIELDS.values()]
    rows = {values[0]: values[1:] for _, row in df[columns].iterrows() for values in zip(*map(split_list, row))}
    return pd.DataFrame.from_dict(rows, orient='index', columns=list(INST_CONTEXT_FIELDS))

def classify_inst_batch(batch_df, max_tokens=1024):
    ''' Classify a batch of unique institutions with Claude Haiku 5.5. Returns {institution id: group} '''
    contexts = [{"key": key, **{field: None if value == MISSING else value for field, value in row.items()}}
                for key, (_, row) in enumerate(batch_df.iterrows())]
    response = client.messages.parse(
        model="claude-haiku-5-5",
        max_tokens=max_tokens,
        system=INST_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": json.dumps(contexts, ensure_ascii=False)}],
        output_format=InstGroupBatch,
    )
    if response.stop_reason != "end_turn" or response.parsed_output is None:
        print(f"Warning: batch not classified (stop_reason: {response.stop_reason})")
        return {}, response.usage
    inst_ids = batch_df.index
    groups = {inst_ids[inst.key]: inst.inst_group for inst in response.parsed_output.institutions
              if 0 <= inst.key < len(inst_ids)}
    return groups, response.usage

In [80]:
inst_table = build_inst_table(studies_df)
print("Unique institutions:", len(inst_table))

batch_size = 100
max_tokens = 50*batch_size # 50 tokens per institution
inst_groups = {}
input_tokens, output_tokens = 0, 0

for start in range(0, len(inst_table), batch_size):
    try:
        groups, usage = classify_inst_batch(inst_table.iloc[start:start + batch_size], max_tokens)
    except (anthropic.APIStatusError, anthropic.APIConnectionError) as e:
        print(f"Error in batch {start}: {e}")
        continue
    inst_groups.update(groups)
    input_tokens += usage.input_tokens
    output_tokens += usage.output_tokens

inst_table['inst_group'] = inst_table.index.map(inst_groups)
print("Institutions without inst_group:", inst_table['inst_group'].isna().sum())
print("Tokens (input / output):", input_tokens, "/", output_tokens)
print("Estimated cost:", input_tokens/1_000_000 * 0.1 + output_tokens/1_000_000*0.5)

# One group per institution, aligned with the institutions columns (MISSING if not classified)
studies_df['authorships.institutions.inst_group'] = studies_df['authorships.institutions.id'].map(
    lambda inst_ids: join_list([inst_groups.get(inst_id, MISSING) for inst_id in split_list(inst_ids)]))
assert (group_counts(studies_df, INST_COLS + ['authorships.institutions.inst_group']).nunique(axis=1) == 1).all(), "Misaligned inst_group"

Unique institutions: 838
Institutions without inst_group: 0
Tokens (input / output): 57305 / 27382
Estimated cost: 0.0194215


In [81]:
# Unique institutions per group
print(inst_table['inst_group'].value_counts(dropna=False))

# Studies per group (each group counted once per study, studies without institutions as MISSING)
studies_df['authorships.institutions.inst_group'].fillna(MISSING).str.split(';').map(set).explode().value_counts()

inst_group
LAC                           278
USA & Canada                  228
Europe                        194
Other institutions             89
Multilateral organizations     33
Honduras                       16
Name: count, dtype: int64


authorships.institutions.inst_group
Honduras                      377
Not indicated                 289
LAC                           258
USA & Canada                  226
Europe                        170
Multilateral organizations    135
Other institutions             55
Name: count, dtype: int64

## Formatos de salida

Script principal de estudios

In [99]:
studies_df_audit = studies_df.copy()

In [100]:

for int_var in ['thematic_axis', 'pillar', 'strategic_objective']:
    studies_df_audit[int_var] = studies_df_audit['thematic_id'].apply(
        lambda value: ";".join(
            set(
                [thematics_df[thematics_df['thematic_id'] == item][int_var].iloc[0] 
                    for item in split_list(value)]
            )
        ) if value else None
    )

In [104]:
studies_df_audit[['thematic_id', 'thematic_axis', 'pillar', 'strategic_objective']].sample(5)

,thematic_id,thematic_axis,pillar,strategic_objective
621,331,Mercado laboral,3. Promover la integración económica y la prod...,3.3. Promover vías hacia el empleo formal
1162,121;123,Protección social;Salud,1. Mejorar la prestación de servicios públicos,1.2. Reforzar el capital humano y la protecció...
81,321,Productividad empresarial y financiamiento,3. Promover la integración económica y la prod...,3.2. Mejorar el rendimiento de la empresa y el...
134,311,Sistema eléctrico,3. Promover la integración económica y la prod...,3.1. Promover infraestructuras productivas y c...
967,331,Mercado laboral,3. Promover la integración económica y la prod...,3.3. Promover vías hacia el empleo formal


In [106]:
studies_df_audit.to_excel("bibliometrics_honduras.xlsx", index=False)

### Estructura de base de datos

`studies_df` se separa en tablas relacionadas por sus llaves:

| Tabla | Llave | Contenido |
|---|---|---|
| `studies` | `study_id` | Una fila por estudio: metadatos, abstract, publisher, autores y citaciones. |
| `institutions` | `institution_id` | Una fila por institución: nombre, país, tipo, organización raíz (`root_display_name`) y grupo (`inst_group`). |
| `study_institutions` | `study_id`, `institution_id` | Tabla puente estudio–institución. Los estudios sin instituciones no tienen filas. |
| `topics` | `topic_id` | Catálogo de tópicos de OpenAlex con su subcampo, campo y dominio. |
| `study_topics` | `study_id`, `topic_id` | Tabla puente estudio–tópico (`primary_topic`). |
| `thematics` | `thematic_id` | Ejes temáticos de `queries_honduras_ET.json`, con sus keywords y la OQL. |
| `study_thematics` | `study_id`, `thematic_id` | Tabla puente estudio–eje temático (un estudio puede tener varios ejes). |

- Los ids de OpenAlex se guardan sin el prefijo `https://openalex.org/`. Los de tópicos, subcampos, campos y dominios se guardan como número (`T10795` → `10795`), igual que en `subfields_tagged.xlsx`.
- El catálogo de tópicos se descarga del snapshot público de OpenAlex en S3 (gratuito, sin API key y sin consumir presupuesto). Se guarda un respaldo en `honduras_backup/topics_snapshot.json`.
- Las columnas `primary_topic.*` de `studies_df` no pasan a `studies`, pero se usan para comprobar que las asignaciones coinciden con el catálogo.
- No se incluyen las columnas de trazabilidad de las imputaciones (`institutions_source`, `publisher_source`, `language_imputed`, `abstract_source`).

In [107]:
# Studies table: entities move to their own tables, imputation trace columns are dropped
topic_cols = [column for column in studies_df.columns if column.startswith('primary_topic.')]
INST_TABLE_COLS = INST_COLS + ['authorships.institutions.inst_group']
source_cols = ['institutions_source', 'publisher_source', 'language_imputed', 'abstract_source']

studies = studies_df.drop(columns=['thematic_id'] + topic_cols + INST_TABLE_COLS + source_cols).\
    rename(columns={'id': 'study_id'})
studies['study_id'] = studies['study_id'].map(short_id)
assert studies['study_id'].is_unique, "Duplicated study ids"
study_ids = studies['study_id'] # Same index as studies_df
print("Studies:", studies.shape)

Studies: (1260, 22)


Instituciones

In [108]:
# Bridge: one row per study and institution
study_institutions = pd.DataFrame(
    [(study_id, short_id(inst_id)) for study_id, inst_ids in zip(study_ids, studies_df['authorships.institutions.id'])
     for inst_id in split_list(inst_ids)],
    columns=['study_id', 'institution_id']).drop_duplicates(ignore_index=True)
assert not study_institutions['institution_id'].eq(MISSING).any(), "Institutions without id"

# Institutions: one row per id from the aligned institutions columns
inst_rows = pd.DataFrame(
    [values for _, row in studies_df[INST_TABLE_COLS].iterrows() for values in zip(*map(split_list, row))],
    columns=[column.rsplit('.', 1)[-1] for column in INST_TABLE_COLS]).\
        rename(columns={'id': 'institution_id'})
inst_rows['institution_id'] = inst_rows['institution_id'].map(short_id)
assert (inst_rows.groupby('institution_id').nunique() == 1).all().all(), "Institution ids with more than one value"
institutions = inst_rows.drop_duplicates('institution_id', ignore_index=True)
assert set(institutions['institution_id']) == set(study_institutions['institution_id'])

print("Institutions:", institutions.shape, "| study_institutions:", study_institutions.shape)
print("Studies without institutions:", (~study_ids.isin(study_institutions['study_id'])).sum())

Institutions: (838, 6) | study_institutions: (1880, 2)
Studies without institutions: 289


Ejes temáticos

In [109]:
thematics = thematics_df.copy()

# Bridge: one row per study and thematic axis ("210;211" -> two rows)
study_thematics = pd.DataFrame(
    {'study_id': study_ids, 'thematic_id': studies_df['thematic_id'].astype(str).str.split(';')}).\
    explode('thematic_id').drop_duplicates(ignore_index=True)
assert study_thematics['thematic_id'].isin(thematics['thematic_id']).all(), "Unknown thematic ids"

print("Thematics:", thematics.shape, "| study_thematics:", study_thematics.shape)
print("Studies with more than one thematic:", (study_thematics['study_id'].value_counts() > 1).sum())

Thematics: (20, 8) | study_thematics: (1307, 2)
Studies with more than one thematic: 45


Tópicos: catálogo del snapshot público de OpenAlex y validación de las asignaciones

In [110]:
import gzip

TOPICS_MANIFEST = "https://openalex.s3.amazonaws.com/data/jsonl/topics/manifest.json" # Public snapshot: no API key or cost

TOPIC_FIELDS = ['id', 'display_name', 'description', 'legacy_keywords', 'ids', 'subfield', 'field', 'domain',
                'works_count', 'updated_date'] # Fields kept in the backup (the full records weigh ~40 MB)

num_id = lambda openalex_id: int(str(openalex_id).rsplit('/', 1)[-1].lstrip('T')) # T10795 -> 10795, .../subfields/1800 -> 1800

def fetch_topics_snapshot(path):
    ''' OpenAlex topics from the public S3 snapshot as {meta, results} (read from the backup if it exists) '''
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    manifest = requests.get(TOPICS_MANIFEST, timeout=60).json()
    records = []
    for file in manifest['files']:
        req = requests.get(file['url'].replace('s3://openalex/', 'https://openalex.s3.amazonaws.com/'), timeout=300)
        req.raise_for_status()
        records += [{k: v for k, v in json.loads(line).items() if k in TOPIC_FIELDS}
                    for line in gzip.decompress(req.content).decode('utf-8').splitlines() if line]
    assert len(records) == manifest['record_count'], "Incomplete topics snapshot"
    snapshot = {'meta': {k: v for k, v in manifest.items() if k != 'files'}, 'results': records}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(snapshot, f, ensure_ascii=False)
    return snapshot

topics_snapshot = fetch_topics_snapshot(os.path.join(backup_dir, "topics_snapshot.json"))
print("Snapshot date:", topics_snapshot['meta']['date'])

# Same columns as the OpenAlex topic mapping table
topics = pd.DataFrame([{
    'topic_id': num_id(topic['id']), 
    'topic_name': topic['display_name'],
    **{f'{level}_{field}': (num_id(topic[level]['id']) if field == 'id' else topic[level]['display_name'])
       for level in ['subfield', 'field', 'domain'] 
       for field in ['id', 'name']},
    'keywords': topic.get('legacy_keywords'), 
    'summary': topic.get('description'),
    'wikipedia_url': (topic.get('ids') or {}).get('wikipedia'),
    'works_count': topic.get('works_count'), 'updated_date': topic.get('updated_date'),
    } 
    for topic in topics_snapshot['results']]
    ).sort_values('topic_id', ignore_index=True)
assert topics['topic_id'].is_unique, "Duplicated topic ids"
print("Topics:", topics.shape)

Snapshot date: 2026-10-14
Topics: (4516, 13)


In [111]:
study_topics = pd.DataFrame(
    {'study_id': study_ids, 'topic_id': studies_df['primary_topic.id'].map(num_id)})
assert study_topics['topic_id'].isin(topics['topic_id']).all(), "Study topics not in the catalog"
print("study_topics:", study_topics.shape)

# Check the topic assignments of the studies against the catalog (OpenAlex may update the taxonomy)
topic_pairs = {'primary_topic.display_name': 'topic_name',
               'primary_topic.subfield.id': 'subfield_id', 'primary_topic.subfield.display_name': 'subfield_name',
               'primary_topic.field.id': 'field_id', 'primary_topic.field.display_name': 'field_name',
               'primary_topic.domain.id': 'domain_id', 'primary_topic.domain.display_name': 'domain_name'}
topic_check = studies_df[list(topic_pairs)].assign(topic_id=study_topics['topic_id']).merge(topics, on='topic_id', how='left')
topic_diffs = pd.DataFrame({column: (topic_check[column].map(num_id) if column.endswith('.id') else topic_check[column]).ne(topic_check[name])
                            for column, name in topic_pairs.items()})
print("Studies with differences:", topic_diffs.any(axis=1).sum())
topic_diffs.sum().rename('differences').to_frame()

study_topics: (1260, 2)
Studies with differences: 0


,differences
primary_topic.display_name,0
primary_topic.subfield.id,0
primary_topic.subfield.display_name,0
primary_topic.field.id,0
primary_topic.field.display_name,0
primary_topic.domain.id,0
primary_topic.domain.display_name,0


Exportar una hoja por tabla

In [112]:
db_tables = {'studies': studies, 'institutions': institutions, 'study_institutions': study_institutions,
             'topics': topics, 'study_topics': study_topics,
             'thematics': thematics, 'study_thematics': study_thematics}

with pd.ExcelWriter("bibliometrics_honduras_db.xlsx") as writer:
    for name, table in db_tables.items():
        table.to_excel(writer, sheet_name=name, index=False)
        print(f"{name}:", table.shape)

studies: (1260, 22)
institutions: (838, 6)
study_institutions: (1880, 2)
topics: (4516, 13)
study_topics: (1260, 2)
thematics: (20, 8)
study_thematics: (1307, 2)
